# Analise exploratoria -- Continua

Dia 3 do desafio: analise exploratoria sobre `data/processed/municipio_mes.csv`,
o dataset agregado municipio x mes gerado pelo pipeline em `etl/` a partir dos
18.926.623 eventos reais de interrupcao publicados pela ANEEL (2024 + 2025).

Perguntas que este notebook responde:

1. Como o volume de interrupcoes varia ao longo do ano (sazonalidade)?
2. Como o risco se distribui entre regioes do Brasil?
3. Quais causas dominam as interrupcoes -- e quao granular e essa informacao de fato?
4. Quais municipios tem o maior risco historico (indicadores aproximados de FEC/DEC)?
5. Os indicadores sao correlacionados entre si de um jeito que ajude (ou atrapalhe) o modelo do Dia 4?

Ver `docs/DEVLOG.md` (Dia 3) para o resumo das conclusoes e `docs/REQUISITOS.md`
para as decisoes de produto por tras dos indicadores aproximados (`fec_aprox`,
`dec_aprox_horas`).


In [1]:
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

ARTIFACTS = r"/home/claude/radar-continuidade/ml/artifacts"

df = pd.read_parquet(r"/home/claude/radar-continuidade/data/processed/municipio_mes.parquet")
print(df.shape)
df.head(3)


(131793, 62)


,codigo_ibge_resolvido,nome_municipio,uf_sigla,regiao,ano,mes,n_eventos_total,n_eventos_programados,n_eventos_validos,consumidores_ativos_max,...,causa_interno,causa_nao_informado,causa_nr_cancelada,causa_programada,causa_proprias_do_sistema,duracao_total_horas,n_municipios_no_conjunto,n_distribuidoras,fec_aprox,dec_aprox_horas
0,1100015,Alta Floresta D'Oeste,RO,Norte,2024,1,92.388889,2.166667,90.222222,4575.666667,...,0.0,0.0,0.0,0.0,0.0,18531.353333,12.0,1,0.019718,4.049979
1,1100015,Alta Floresta D'Oeste,RO,Norte,2024,2,77.333333,1.388889,75.944444,4559.166667,...,0.0,0.0,0.0,0.0,0.0,14914.135833,12.0,1,0.016658,3.271242
2,1100015,Alta Floresta D'Oeste,RO,Norte,2024,3,80.944444,2.055556,78.888889,4513.333333,...,0.0,0.0,0.0,0.0,0.0,13627.337500,12.0,1,0.017479,3.019351


## 1. Visao geral e qualidade dos dados

In [2]:
df.dtypes.value_counts()


float64    55
str         4
int32       2
int64       1
Name: count, dtype: int64

In [3]:
# quantos municipios e meses distintos temos
print("municipios distintos (codigo_ibge_resolvido):", df["codigo_ibge_resolvido"].nunique())
print("meses distintos:", df[["ano", "mes"]].drop_duplicates().shape[0], "(2024-01 a 2025-12)")
print("linhas totais:", len(df))


municipios distintos (codigo_ibge_resolvido): 5514
meses distintos: 24 (2024-01 a 2025-12)
linhas totais: 131793


In [4]:
# nulos -- so existem em nome_municipio/uf_sigla/regiao, e so quando o conjunto
# nao teve correspondencia na bridge conjunto->municipio da ANEEL (ver etl/ibge.py)
nulos = df.isna().sum()
nulos = nulos[nulos > 0]
print(nulos)

sem_bridge = df[df["nome_municipio"].isna()]
conjuntos_sem_bridge = sem_bridge["codigo_ibge_resolvido"].nunique()
print(f"\n{len(sem_bridge)} linhas ({len(sem_bridge)/len(df):.3%} do total) "
      f"vem de {conjuntos_sem_bridge} conjuntos sem correspondencia na bridge "
      "'IndQual Municipio' -- ficam em grupo proprio (codigo_ibge_resolvido "
      "= 'CONJUNTO_<id>'), sem regiao/UF conhecida, mas SEM ser descartados "
      "da agregacao (decisao de projeto, ver docs/REQUISITOS.md).")


nome_municipio    120
uf_sigla          120
regiao            120
dtype: int64

120 linhas (0.091% do total) vem de 5 conjuntos sem correspondencia na bridge 'IndQual Municipio' -- ficam em grupo proprio (codigo_ibge_resolvido = 'CONJUNTO_<id>'), sem regiao/UF conhecida, mas SEM ser descartados da agregacao (decisao de projeto, ver docs/REQUISITOS.md).


**Conclusao:** a cobertura da bridge conjunto->municipio e excelente (>99,9% das
linhas tem municipio/UF/regiao resolvidos). O pequeno residuo sem correspondencia
fica isolado em seu proprio grupo -- nao contamina nenhum municipio real, e nao
pode ser usado no ranking por municipio (sera excluido do treino do modelo por
nao ter uma chave geografica valida).

## 2. Sazonalidade -- volume de eventos por mes

In [5]:
por_mes = df.groupby(["ano", "mes"], as_index=False)["n_eventos_total"].sum()
por_mes["periodo"] = por_mes["ano"].astype(str) + "-" + por_mes["mes"].astype(str).str.zfill(2)

fig, ax = plt.subplots(figsize=(11, 4.5))
for ano, grupo in por_mes.groupby("ano"):
    ax.plot(grupo["mes"], grupo["n_eventos_total"], marker="o", label=str(ano))
ax.set_xticks(range(1, 13))
ax.set_xlabel("Mes")
ax.set_ylabel("N. de eventos (ponderado por fan-out)")
ax.set_title("Volume nacional de interrupcoes por mes -- 2024 vs 2025")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e3:.0f} mil"))
ax.legend(title="Ano")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(f"{ARTIFACTS}/sazonalidade_mensal.png", dpi=110)
plt.show()


In [6]:
# variacao pico-vale, como % da media -- para quantificar a sazonalidade
media = por_mes["n_eventos_total"].mean()
pico = por_mes["n_eventos_total"].max()
vale = por_mes["n_eventos_total"].min()
print(f"media mensal: {media:,.0f} eventos")
print(f"pico: {pico:,.0f} ({(pico/media - 1):+.1%} vs media)")
print(f"vale: {vale:,.0f} ({(vale/media - 1):+.1%} vs media)")


media mensal: 788,609 eventos
pico: 1,015,726 (+28.8% vs media)
vale: 579,843 (-26.5% vs media)


**Conclusao:** ha um padrao sazonal claro e consistente nos dois anos: os meses
de verao/chuvas no Brasil (dezembro-janeiro, e um pico secundario em
setembro-outubro) concentram mais eventos, com um vale bem definido em
junho-julho (inverno/seco). O padrao se repete quase identico entre 2024 e 2025
-- e o argumento mais forte a favor de um baseline de **persistencia sazonal**
(prever o mes usando o mesmo mes do ano anterior) no lugar de so persistencia do
mes anterior. Isso vira `mes` (ciclico) como feature obrigatoria no Dia 4.

## 3. Distribuicao regional

In [7]:
por_regiao = df.groupby("regiao").agg(
    n_eventos=("n_eventos_total", "sum"),
    consumidores=("consumidores_ativos_max", "sum"),
    municipios=("codigo_ibge_resolvido", "nunique"),
).sort_values("n_eventos", ascending=False)
por_regiao["eventos_por_1000_consumidores_mes"] = (
    por_regiao["n_eventos"] / por_regiao["consumidores"] * 1000 / 24
)
por_regiao


,n_eventos,consumidores,municipios,eventos_por_1000_consumidores_mes
regiao,,,,
Sudeste,6.898528e+06,9.462268e+08,1667,0.303774
Nordeste,4.276541e+06,5.746635e+08,1789,0.310076
Sul,3.167294e+06,3.308376e+08,1141,0.398898
Centro-Oeste,2.555791e+06,1.800089e+08,467,0.591589
Norte,1.987516e+06,1.432288e+08,445,0.578188


In [8]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

ordem = por_regiao.index
axes[0].bar(ordem, por_regiao["n_eventos"] / 1e6)
axes[0].set_title("Total de eventos (2024-2025)")
axes[0].set_ylabel("Milhoes de eventos")
axes[0].tick_params(axis="x", rotation=20)

axes[1].bar(ordem, por_regiao["eventos_por_1000_consumidores_mes"], color="darkorange")
axes[1].set_title("Eventos por 1.000 consumidores / mes\n(normalizado pelo tamanho da regiao)")
axes[1].set_ylabel("Eventos / 1.000 consumidores / mes")
axes[1].tick_params(axis="x", rotation=20)

fig.tight_layout()
fig.savefig(f"{ARTIFACTS}/distribuicao_regional.png", dpi=110)
plt.show()


**Conclusao:** em volume absoluto o Sudeste domina (mais consumidores). Mas
normalizando por consumidor -- a mesma logica do FEC regulatorio -- o quadro
muda: Norte e Nordeste tem taxa de eventos por consumidor sensivelmente maior
que Sudeste/Sul. Isso confirma que a normalizacao usada em `fec_aprox` (e nao o
volume bruto de eventos) e a metrica certa para comparar risco entre municipios
de tamanhos muito diferentes -- ranquear por volume bruto favoreceria sempre
cidades grandes, nao as de fato mais arriscadas.

## 4. Mix de causas

In [9]:
causa_cols = [c for c in df.columns if c.startswith("causa_")]
total_validos = df["n_eventos_validos"].sum()
soma_causas = df[causa_cols].sum()

# confirma que as colunas de causa sao mutuamente exclusivas (cada evento
# valido cai em exatamente uma) -- a soma bate com n_eventos_validos.
print("soma de todas as colunas de causa:", soma_causas.sum())
print("n_eventos_validos (total):", total_validos)


soma de todas as colunas de causa: 17924402.0
n_eventos_validos (total): 17924402.0


In [10]:
top_causas = (soma_causas / total_validos * 100).sort_values(ascending=False).head(12)

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.barh(top_causas.index[::-1], top_causas.values[::-1], color="steelblue")
ax.set_xlabel("% dos eventos validos")
ax.set_title("Causas mais frequentes (DscFatoGeradorInterrupcao normalizado)")
fig.tight_layout()
fig.savefig(f"{ARTIFACTS}/mix_de_causas.png", dpi=110)
plt.show()

top_causas.round(2)


causa_interna                                                                                88.18
causa_interno                                                                                 6.80
causa_interna/nao_programada/proprias_do_sistema/falha_de_material_ou_equipamento             1.32
causa_interna/nao_programada/meio_ambiente/arvore_ou_vegetacao                                0.93
causa_interna/nao_programada/nao_classificada                                                 0.58
causa_interna/nao_programada/proprias_do_sistema/nao_identificada                             0.37
causa_interna/nao_programada/meio_ambiente/descarga_atmosferica                               0.34
causa_interna/nao_programada/meio_ambiente/animais                                            0.34
causa_externa                                                                                 0.22
causa_interna/nao_programada/meio_ambiente/vento                                              0.21
causa_inte

In [11]:
generico = (soma_causas.get("causa_interna", 0) + soma_causas.get("causa_interno", 0)) / total_validos
print(f"{generico:.1%} dos eventos validos so tem a causa generica "
      "'interna'/'interno', sem nenhum nivel de detalhe alem disso.")
print(f"Existem {len(causa_cols)} strings de causa distintas no total, mas so "
      "uma duzia tem peso pratico -- a cauda longa e irrelevante para o modelo.")


95.0% dos eventos validos so tem a causa generica 'interna'/'interno', sem nenhum nivel de detalhe alem disso.
Existem 46 strings de causa distintas no total, mas so uma duzia tem peso pratico -- a cauda longa e irrelevante para o modelo.


**Conclusao (importante para nao superestimar o dado):** ~95% dos eventos
validos so tem a causa generica "interna"/"interno" -- sem nenhuma
granularidade (arvore, animal, equipamento, etc.). So uma minoria dos registros
tem causa detalhada. Isso significa que "causas dominantes" como feature
individual tem pouco poder preditivo pratico hoje -- o campo esta mais para
"a interrupcao foi causada pela propria rede/distribuidora ou por algo
externo" do que para um diagnostico de causa raiz. Documentado como limitacao
de dado (nao um bug do pipeline) -- ver `etl/README.md`.

## 5. Ranking de risco historico por municipio

In [12]:
por_municipio = df.dropna(subset=["nome_municipio"]).groupby(
    ["codigo_ibge_resolvido", "nome_municipio", "uf_sigla", "regiao"], as_index=False
).agg(
    meses_com_dado=("ano", "count"),
    fec_aprox_medio=("fec_aprox", "mean"),
    dec_aprox_medio=("dec_aprox_horas", "mean"),
    n_eventos_medio=("n_eventos_total", "mean"),
    consumidores=("consumidores_ativos_max", "max"),
)

# so municipios com historico completo (24 meses) entram no ranking --
# historico incompleto (municipio so aparece em alguns meses) enviesaria a
# media para cima ou para baixo sem base de comparacao justa.
completos = por_municipio[por_municipio["meses_com_dado"] == 24]
print(f"{len(completos)} de {len(por_municipio)} municipios tem os 24 meses completos "
      f"({len(completos)/len(por_municipio):.1%}).")


4966 de 5509 municipios tem os 24 meses completos (90.1%).


In [13]:
top15_fec = completos.sort_values("fec_aprox_medio", ascending=False).head(15)
top15_fec[["nome_municipio", "uf_sigla", "regiao", "fec_aprox_medio", "dec_aprox_medio", "n_eventos_medio"]]


,nome_municipio,uf_sigla,regiao,fec_aprox_medio,dec_aprox_medio,n_eventos_medio
5481,São Miguel do Araguaia,GO,Centro-Oeste,0.054681,2.195306,199.850926
5297,Bonópolis,GO,Centro-Oeste,0.052766,2.677833,168.118750
5498,Uirapuru,GO,Centro-Oeste,0.048941,3.597046,113.131283
5415,Mundo Novo,GO,Centro-Oeste,0.048796,2.737126,212.797950
5383,Itarumã,GO,Centro-Oeste,0.044976,3.480279,137.817758
405,Piraquê,TO,Norte,0.044366,2.169030,26.175000
5281,Aparecida do Rio Doce,GO,Centro-Oeste,0.043974,4.404015,88.676091
358,Fátima,TO,Norte,0.043776,3.320951,58.278409
337,Brejinho de Nazaré,TO,Norte,0.043776,3.320951,58.278409
394,Oliveira de Fátima,TO,Norte,0.043776,3.320951,58.278409


In [14]:
fig, ax = plt.subplots(figsize=(9, 5.5))
labels = top15_fec["nome_municipio"] + "/" + top15_fec["uf_sigla"]
ax.barh(labels[::-1], top15_fec["fec_aprox_medio"][::-1], color="firebrick")
ax.set_xlabel("FEC aproximado medio (2024-2025)")
ax.set_title("Top 15 municipios por risco historico de interrupcao")
fig.tight_layout()
fig.savefig(f"{ARTIFACTS}/top15_risco_historico.png", dpi=110)
plt.show()


In [15]:
# o risco historico e estavel mes a mes para o mesmo municipio, ou e ruido?
# correlacao entre fec_aprox de um municipio num mes e no mes seguinte.
painel = df.dropna(subset=["nome_municipio"]).sort_values(["codigo_ibge_resolvido", "ano", "mes"]).copy()
painel["fec_aprox_mes_seguinte"] = painel.groupby("codigo_ibge_resolvido")["fec_aprox"].shift(-1)
valido = painel.dropna(subset=["fec_aprox_mes_seguinte"])
corr_persistencia = valido["fec_aprox"].corr(valido["fec_aprox_mes_seguinte"])
print(f"correlacao entre fec_aprox(mes) e fec_aprox(mes+1) para o mesmo municipio: {corr_persistencia:.2f}")


correlacao entre fec_aprox(mes) e fec_aprox(mes+1) para o mesmo municipio: 0.84


**Conclusao:** o risco de interrupcao e fortemente persistente por municipio --
quem teve FEC alto num mes tende a ter FEC alto no mes seguinte (correlacao alta
mes a mes). Isso e uma otima noticia para a viabilidade do produto: existe sinal
historico real para ranquear risco, nao e ruido puro. Tambem estabelece a barra
que qualquer modelo do Dia 4 precisa vencer -- ver a analise de baseline em
`02-baseline.ipynb`.

## 6. Correlacao entre indicadores

In [16]:
cols_indicadores = [
    "n_eventos_total", "n_eventos_validos", "consumidores_ativos_max",
    "duracao_total_horas", "n_distribuidoras", "n_municipios_no_conjunto",
    "fec_aprox", "dec_aprox_horas",
]
corr = df[cols_indicadores].corr()

fig, ax = plt.subplots(figsize=(7.5, 6.5))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(cols_indicadores)))
ax.set_yticks(range(len(cols_indicadores)))
ax.set_xticklabels(cols_indicadores, rotation=45, ha="right")
ax.set_yticklabels(cols_indicadores)
for i in range(len(cols_indicadores)):
    for j in range(len(cols_indicadores)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("Correlacao entre indicadores (municipio x mes)")
fig.tight_layout()
fig.savefig(f"{ARTIFACTS}/correlacao_indicadores.png", dpi=110)
plt.show()


**Conclusao:** `fec_aprox` e `dec_aprox_horas` sao correlacionados mas nao
identicos (frequencia e duracao medem coisas diferentes -- um municipio pode
ter muitas interrupcoes curtas ou poucas interrupcoes longas). `n_eventos_total`
e fortemente ligado a `consumidores_ativos_max` (municipios maiores tem mais
eventos em volume bruto, reforcando a decisao da secao 3 de normalizar por
consumidor). `n_municipios_no_conjunto` (grau de compartilhamento do conjunto,
usado no fan-out) nao tem correlacao forte com o risco em si -- e uma variavel
estrutural do dado, nao um sinal de risco.

## Resumo executivo (Dia 3 -- EDA)

1. **Sazonalidade real e consistente** entre os dois anos (pico
   dezembro-janeiro/setembro-outubro, vale junho-julho) -> `mes` deve entrar
   como feature ciclica no modelo.
2. **Risco bruto favorece cidades grandes; risco normalizado (FEC/DEC
   aproximados) inverte o quadro** -> confirma a escolha de normalizar por
   consumidor em vez de usar contagem bruta de eventos como alvo do modelo.
3. **Causa da interrupcao e um campo pouco granular na pratica** (~95% caem em
   "interna" generico) -> util como feature binaria grosseira (interna vs.
   externa vs. programada), nao como taxonomia detalhada de causa raiz.
4. **O risco por municipio e persistente mes a mes** -> existe sinal real para
   ranquear, e um baseline de persistencia (ingenuo) e um piso de comparacao
   dificil de bater, nao um adversario fraco -- ver `02-baseline.ipynb`.
5. **Cobertura de dados e muito boa** (>99,9% das linhas com municipio
   resolvido); o pequeno residuo sem correspondencia na bridge fica isolado e
   documentado, sem contaminar o ranking dos municipios reais.
